# Set up

In [11]:
import pandas as pd
import os
import matplotlib as plt

from surprise import Reader, AlgoBase, Dataset, accuracy
from surprise.model_selection import train_test_split
import numpy as np
from collections import defaultdict
import recmetrics
from sklearn.metrics import ndcg_score

In [12]:
SEED = 42; TOP_K = 10; RELEVANCE_THRESHOLD = 7; MIN_ITEM_RATINGS = 50
N_USERS = 5000; N_USERS_COVERAGE = 50

np.random.seed(SEED)
ratings_df = pd.read_csv("../processed_data/explicit_ratings.csv")
all_users = ratings_df["user_id"].unique()
np.random.shuffle(all_users)

reader      = Reader(rating_scale=(1, 10))
reader_wide = Reader(rating_scale=(1, 20))

sample = ratings_df[ratings_df["user_id"].isin(all_users[:N_USERS])][["user_id", "anime_id", "rating"]]
data = Dataset.load_from_df(sample, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=SEED)

sample_cov = ratings_df[ratings_df["user_id"].isin(all_users[:N_USERS_COVERAGE])][["user_id", "anime_id", "rating"]]
data_cov = Dataset.load_from_df(sample_cov, reader_wide)
trainset_cov = data_cov.build_full_trainset()
antitestset_cov = trainset_cov.build_anti_testset()
catalog = [trainset_cov.to_raw_iid(iid) for iid in trainset_cov.all_items()]

genre_cols = ['Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Harem', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic', 'Martial Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody', 'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi', 'Seinen', 'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai', 'Slice of Life', 'Space', 'Sports', 'Super Power', 'Supernatural', 'Thriller', 'Vampire', 'Yaoi']
anime_meta = pd.read_csv("../processed_data/anime_processed.csv")[["MAL_ID"] + genre_cols].set_index("MAL_ID")


In [13]:
import gc

def get_top_n(predictions, n=TOP_K):
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est, true_r))
    return {uid: sorted(v, key=lambda x: x[1], reverse=True)[:n] for uid, v in user_preds.items()}

def eval_accuracy(predictions, n=TOP_K, threshold=RELEVANCE_THRESHOLD):
    rmse = accuracy.rmse(predictions, verbose=False)
    mae  = accuracy.mae(predictions, verbose=False)
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est, true_r))
    precisions, recalls, ndcgs = [], [], []
    for uid, preds in user_preds.items():
        top = sorted(preds, key=lambda x: x[1], reverse=True)[:n]
        hits      = sum(r >= threshold for _, _, r in top)
        total_rel = sum(r >= threshold for _, _, r in preds)
        precisions.append(hits / n)
        recalls.append(hits / total_rel if total_rel > 0 else 0)
        true_r = [r for _, _, r in top]; est_r = [e for _, e, _ in top]
        if len(true_r) > 1:
            ndcgs.append(ndcg_score([true_r], [est_r]))
    return {"RMSE": round(rmse, 4), "MAE": round(mae, 4),
            f"Precision@{n}": round(np.mean(precisions), 4),
            f"Recall@{n}":    round(np.mean(recalls), 4),
            f"NDCG@{n}":      round(np.mean(ndcgs), 4)}

def eval_coverage(predictions, catalog, feature_df, n=TOP_K):
    top_n     = get_top_n(predictions, n)
    full_recs = [[iid for iid, _, _ in v] for v in top_n.values() if len(v) == n]
    cov  = recmetrics.prediction_coverage(full_recs, catalog) if full_recs else 0.0
    pers = recmetrics.personalization(full_recs)              if full_recs else 0.0
    ils  = recmetrics.intra_list_similarity(full_recs, feature_df)
    return {"Coverage": round(cov, 4), "Personalization": round(pers, 4),
            "Intra-List Similarity": round(ils, 4)}

# Non-Personalized Recommenders

## Popular Recommender

In [14]:
class PopularRecommender(AlgoBase):
    def __init__(self, min_ratings=MIN_ITEM_RATINGS):
        AlgoBase.__init__(self)
        self.min_ratings = min_ratings

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        df = pd.DataFrame([(i, r) for (_, i, r) in trainset.all_ratings()], columns=["item", "rating"])
        stats = df.groupby("item").agg(count=("rating", "count"), mean=("rating", "mean"))
        self.item_means = stats.loc[stats["count"] >= self.min_ratings, "mean"]
        self.global_mean = trainset.global_mean
        return self

    def estimate(self, u, i):
        if i in self.item_means.index:
            return self.item_means[i]
        return self.global_mean

In [15]:
popular = PopularRecommender()

popular.fit(trainset)
acc = eval_accuracy(popular.test(testset))

popular.fit(trainset_cov)
cov_preds = popular.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                     1.6112
MAE                      1.2166
Precision@10             0.8144
Recall@10                0.5219
NDCG@10                  0.9735
Coverage                 0.7700
Personalization          0.2209
Intra-List Similarity    0.2851
dtype: float64

## Random Recommender

In [16]:
class RandomRecommender(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        ratings = [r for (_, _, r) in trainset.all_ratings()]
        self.mean = np.mean(ratings)
        self.std  = np.std(ratings)
        return self

    def estimate(self, u, i):
        return np.random.normal(self.mean, self.std)

In [17]:
random_rec = RandomRecommender()

random_rec.fit(trainset)
acc = eval_accuracy(random_rec.test(testset))

random_rec.fit(trainset_cov)
cov_preds = random_rec.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                      2.4038
MAE                       1.9041
Precision@10              0.7183
Recall@10                 0.4844
NDCG@10                   0.9472
Coverage                 12.6600
Personalization           0.9978
Intra-List Similarity     0.1975
dtype: float64

# Content-Based Recommenders

# Collaborative-Filtering RS

In [18]:
import pandas as pd

# Full explicit ratings only. Do NOT replace Surprise `data` (comes from Set up `sample`).
ratings_cf = pd.read_csv(
    "../processed_data/explicit_ratings.csv",
    usecols=["user_id", "anime_id", "rating"],
)
print("ratings_cf loaded. SVD/CV use `data` from Set up (N_USERS subsample).")


In [19]:
print("Rows:", ratings_cf.shape[0])
print("Users:", ratings_cf['user_id'].nunique())
print("Anime:", ratings_cf['anime_id'].nunique())
print("Mean rating:", ratings_cf['rating'].mean())
print("Std rating:", ratings_cf['rating'].std())

Rows: 61103354
Users: 297095
Anime: 15168
Mean rating: 7.449149878090162
Std rating: 1.7517239702004637


Naive baseline

In [ ]:
import numpy as np

global_mean = ratings_cf['rating'].mean()
y_true = ratings_cf['rating'].to_numpy(dtype=np.float64)
y_pred = np.full_like(y_true, global_mean, dtype=np.float64)

print("Naive global mean on full ratings_cf (not the N_USERS Surprise `data` subsample):")
print("RMSE:", float(np.sqrt(np.mean((y_true - y_pred) ** 2))))
print("MAE:", float(np.mean(np.abs(y_true - y_pred))))


Actual CF Model:

In [ ]:
import random
import numpy as np
from surprise import SVD, accuracy
from surprise.model_selection import train_test_split

my_seed = 1234
random.seed(my_seed)
np.random.seed(my_seed)

# `data` = Oskar N_USERS subsample from Set up.
train_cf, test_cf = train_test_split(data, test_size=0.20, random_state=my_seed)

algo = SVD(random_state=my_seed, n_factors=50, n_epochs=10, verbose=True)
algo.fit(train_cf)

predictions = algo.test(test_cf)

accuracy.rmse(predictions)
accuracy.mae(predictions)


Testing model acrros KFolds (k=5)

In [ ]:
from surprise import SVD
from surprise.model_selection import cross_validate, KFold

performance = cross_validate(
    SVD(random_state=my_seed, n_factors=50, n_epochs=10),
    data,
    measures=['RMSE', 'MAE'],
    cv=KFold(n_splits=5, random_state=my_seed, shuffle=True),
    return_train_measures=True,
    verbose=True,
    n_jobs=-1
)


In [ ]:
print("Mean test RMSE:", np.mean(performance['test_rmse']))
print("Mean test MAE:", np.mean(performance['test_mae']))
print("Mean train RMSE:", np.mean(performance['train_rmse']))
print("Mean train MAE:", np.mean(performance['train_mae']))

The CF was trained using explicit feedback only since rating 0 represents unrated items. The naive baseline that always predicts the global mean rating achieved around 1.777 RMSE where as the SCD model roduced the RMSE to about 1.288 and MAE to about 0.957 showing a clear improvement. Cross validation results were consistent across each folds but the lower trainign error compared with the test suggests some overfitting or inherent diffuclty since the dataset is sparse.

## Interpret predictions and generate top 10 recommednations for sample user

In [ ]:
# === modeling.ipynb patch: processed `anime` + `anime_pop` (disk version; Revert File if editor differs) ===
import numpy as np

# Anime metadata from EDA pipeline (cleaned). Rebuild `Genres` string from genre dummies.
_genre_candidates = [
    "Action", "Adventure", "Cars", "Comedy", "Dementia", "Demons", "Drama", "Ecchi", "Fantasy",
    "Game", "Harem", "Historical", "Horror", "Josei", "Kids", "Magic", "Martial Arts", "Mecha",
    "Military", "Music", "Mystery", "Parody", "Police", "Psychological", "Romance", "Samurai",
    "School", "Sci-Fi", "Seinen", "Shoujo", "Shoujo Ai", "Shounen", "Shounen Ai", "Slice of Life",
    "Space", "Sports", "Super Power", "Supernatural", "Thriller", "Vampire", "Yaoi",
]

anime = pd.read_csv("../processed_data/anime_processed.csv")
_genre_cols = [c for c in _genre_candidates if c in anime.columns]
_gm = anime[_genre_cols].fillna(0).to_numpy(dtype=np.int8)
_gn = np.array(_genre_cols)
anime["Genres"] = [", ".join(_gn[row == 1]) for row in _gm]

# Popularity: full explicit_ratings vs Oskar's `sample` (Set up must be run first for `sample`).
USE_SUBSAMPLE_FOR_ANIME_POP = False
_pop_base = sample if USE_SUBSAMPLE_FOR_ANIME_POP else ratings_cf
anime_pop = (
    _pop_base.groupby("anime_id", as_index=False)
    .agg(
        n_interactions=("user_id", "count"),
        n_ratings=("rating", lambda x: (x > 0).sum()),
        mean_rating=("rating", lambda x: x[x > 0].mean() if (x > 0).any() else np.nan),
    )
)
print(
    "anime: anime_processed.csv | anime_pop:",
    "subsample" if USE_SUBSAMPLE_FOR_ANIME_POP else "full explicit_ratings",
)


In [ ]:
predictions[:5]

pick a user with enough ratings

In [ ]:
user_rating_counts = ratings_cf.groupby('user_id').size().sort_values()

print(user_rating_counts.describe())
user_rating_counts[user_rating_counts >= 30].head(10)

In [ ]:
uid = 75051

In [ ]:
user_history = ratings_cf[ratings_cf['user_id'] == uid].copy()

user_history = user_history.merge(
    anime[['MAL_ID', 'Name', 'Genres', 'Type', 'Score', 'Members']],
    left_on='anime_id',
    right_on='MAL_ID',
    how='left'
)

user_history = user_history.sort_values('rating', ascending=False)

user_history[['Name', 'rating', 'Genres', 'Type', 'Score', 'Members']].head(15)

We selected user 75051, who has rated 30 anime.
From their history, we observe that they tend to prefer a mix of genres but mostly Comedy with at times thriller, drama, and mystery.
Their ratings are mostly concentrated around 10-9, suggesting generous behavior.
Additionally, many of the anime they rated have pretty high popualrity with some excuse like Higurashi no Naku Koro, or H2, indicating a preference for mainstream content.

In [ ]:
user_history[['Name', 'rating', 'Genres', 'Type', 'Score', 'Members']].tail(10)

Their lowest rated anime, while still pretty high, suggests that the user dislike series that also have relatively low score, The genre doesnt really change however. Also its worth noting that this user really only looks at TV anime.

What our model recoomends:

In [ ]:
already_rated = set(user_history['anime_id'])
all_anime_ids = set(ratings_cf['anime_id'].unique())

items_to_predict = list(all_anime_ids - already_rated)

In [ ]:
all_predictions = []

for iid in items_to_predict:
    est = algo.predict(uid, iid).est
    all_predictions.append((iid, est))

In [ ]:
top_10 = pd.DataFrame(
    sorted(all_predictions, key=lambda x: x[1], reverse=True)[:10],
    columns=['anime_id', 'predicted_rating']
)

top_10 = top_10.merge(
    anime[['MAL_ID', 'Name', 'Genres', 'Type', 'Score', 'Members']],
    left_on='anime_id',
    right_on='MAL_ID',
    how='left'
)

top_10[['Name', 'predicted_rating', 'Genres', 'Type', 'Score', 'Members']]

In conclusion, the ratings were quite interesting. Some of the pros of the ratings that we could see, that the user actually watched versus what I recommended, are:
- It maintained some of the genres, such as action and drama, with some recommendations including thriller, but for the most part it lacked that thriller/horror aspect that the user likes to watch.
- It mostly recommended TV-type anime, with the exception of only two movies.
- The member counts are quite high in these recommendations, which is good since this user is a bit mainstream with some touch of niche anime.
Some things it could work on are:
- It could try to include more thrillers that the user could like.

This suggests that, while the collaborative-based filtering is a good model for users who have high, good enough ratings, maybe we should combine collaborative filtering with perhaps content-based filtering and create a hybrid recommendation system and see if it improves this.

### Are these recommendations popularity biased?

In [ ]:
top_10_with_pop = top_10.merge(
    anime_pop,
    on='anime_id',
    how='left'
)

top_10_with_pop[['Name', 'predicted_rating', 'n_interactions']]

In [ ]:
print("Avg popularity (recommended):", top_10_with_pop['n_interactions'].mean())
print("Avg popularity (all anime):", anime_pop['n_interactions'].mean())

The CF heavily favors the popular anime. The reccomended items have way more interactions than the overall dataset, which indicates heavy bias toward high-frequency items

### Compare with many users

In [ ]:
sample_users = ratings_cf['user_id'].drop_duplicates().sample(100, random_state=123)

all_top_recs = []

all_anime_ids = set(ratings_cf['anime_id'].unique())

for uid in sample_users:
    seen = set(ratings_cf[ratings_cf['user_id'] == uid]['anime_id'])
    unseen = list(all_anime_ids - seen)

    preds = [(iid, algo.predict(uid, iid).est) for iid in unseen]
    top_preds = sorted(preds, key=lambda x: x[1], reverse=True)[:10]

    for iid, est in top_preds:
        all_top_recs.append((uid, iid, est))

all_top_recs = pd.DataFrame(all_top_recs, columns=['user_id', 'anime_id', 'predicted_rating'])

all_top_recs = all_top_recs.merge(anime_pop, on='anime_id', how='left')

print("Avg popularity of recommendations:", all_top_recs['n_interactions'].mean())
print("Median popularity of recommendations:", all_top_recs['n_interactions'].median())

This shows that this collaborative filtering model that heavily favors popular enemy is not because of the example user being mainstream. It is consistent across all users, not just this single user, which means that the model systematically prioritizes popular enemy when generating recommendations.

### Head vs tail split

In [ ]:
anime_pop_sorted = anime_pop.sort_values('n_interactions', ascending=False).reset_index(drop=True)
anime_pop_sorted['rank_pct'] = (anime_pop_sorted.index + 1) / len(anime_pop_sorted)

def bucket(x):
    if x <= 0.2:
        return 'Head'
    elif x <= 0.5:
        return 'Mid'
    else:
        return 'Tail'

anime_pop_sorted['bucket'] = anime_pop_sorted['rank_pct'].apply(bucket)

all_top_recs = all_top_recs.merge(
    anime_pop_sorted[['anime_id', 'bucket']],
    on='anime_id',
    how='left'
)

all_top_recs['bucket'].value_counts(normalize=True) * 100

This head vs. tail split shows that nearly all recommended items belong to the head of the distribution, which confirms that the model almost entirely ignores niche items.

## Long tail analysis interpretation


Based on the analysis above and from the EDA, as expected, there is a long tail issue going on. The results show strong popularity bias in the collaborative filtering model. The average popularity of the recommended items is way higher than the dataset average, indicating that the model favors widely watched items. This pattern is only shown from the mainstream example guy, but it's also consistent across users. Nearly all recommendations, about 99 of them to be exact, come from the "head items", with almost no representation from the tail.

As mentioned, this behavior is expected due to the long tail distribution of the dataset, where popular anime dominates interactions. As a result, the model prioritizes these items, improving accuracy but limiting diversity. This suggests that maybe a final recommendation system model should be a hybrid one with some mix of random recommendations.